# CNN & Huấn luyện

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import time
import json
from PIL import Image

from data_pipeline import train_loader, test_loader, NUM_CLASSES, test_transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Thiết bị đang sử dụng để huấn luyện: {str(device)}")
print(f"Tổng số class chữ Hán nhận diện: {NUM_CLASSES}")
print(f"Số Batch tập Train: {len(train_loader)} | Tập Test: {len(test_loader)}")

Đang nạp tập Train từ HDF5...
Đang nạp tập Test từ HDF5...
Đã load mapping.json thành công với 7185 chữ Hán.
Tổng số batch trong tập Train: 12023
Tổng số batch trong tập Test: 2899
✅ Sẵn sàng đưa vào file model.ipynb
Thiết bị đang sử dụng để huấn luyện: cuda
Tổng số class chữ Hán nhận diện: 7185
Số Batch tập Train: 12023 | Tập Test: 2899


In [ ]:
torch.backends.cudnn.benchmark = True

class HanziCNN(nn.Module):
    def __init__(self, num_classes):
        super(HanziCNN, self).__init__()
        
        # 64x64 -> 32x32
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=5, padding=2),
            nn.BatchNorm2d(num_features=32), nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # 32x32 -> 16x16
        self.block2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_features=64), nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # 16x16 -> 8x8
        self.block3 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_features=128), nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # 8x8 -> 4x4
        self.block4 = nn.Sequential(
            nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_features=256), nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Lớp GAP biến ma trận 4x4 -> vector 256 chiều
        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(in_features=256, out_features=num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.gap(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

model = HanziCNN(num_classes=NUM_CLASSES).to(device)

✅ Khởi tạo mạng CNN 4 Block thành công!


In [ ]:
import torch.optim.lr_scheduler as lr_scheduler 

EPOCHS = 50 
learning_rate = 0.001

# 7 vòng liên tiếp mà Loss ko giảm thì sẽ Early Stopping
PATIENCE = 7 

# Nếu 2 vòng liên tiếp mà Loss ko giảm thì sẽ giảm Learning Rate
LR_PATIENCE = 2 

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

scheduler = lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min',           # Mục tiêu là theo dõi Loss
    factor=0.5,           # Giảm LR đi 1 nửa
    patience=LR_PATIENCE
)

In [ ]:
print(f"BẮT ĐẦU HUẤN LUYỆN (Tối đa {EPOCHS} Epochs - Tích hợp ReduceLROnPlateau & Early Stopping)...")

# Lưu Model dựa trên Loss thấp nhất trên tập Validation
best_loss = float('inf') 
epochs_no_improve = 0 
best_acc = 0.0 

for epoch in range(EPOCHS):
    start_time = time.time()

    # =========================
    # GIAI ĐOẠN 1: TRAINING
    # =========================
    model.train()
    running_loss = 0.0
    correct_train = total_train = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    train_acc = 100 * correct_train / total_train
    train_loss = running_loss / len(train_loader)

    # =========================
    # GIAI ĐOẠN 2: TESTING (Tính thêm val_loss)
    # =========================
    model.eval()
    test_loss = 0.0 
    correct_test = total_test = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            
            # Tính Val Loss để đưa cho Scheduler và Early Stopping
            loss = criterion(outputs, labels)
            test_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            total_test += labels.size(0)
            correct_test += (predicted == labels).sum().item()

    test_acc = 100 * correct_test / total_test
    val_loss = test_loss / len(test_loader) # Tính trung bình Val Loss của vòng này

    # =========================
    # GIAI ĐOẠN 3: ĐIỀU CHỈNH LR & EARLY STOPPING
    # =========================
    
    # 1. Nạp val_loss cho thuật toán bóp phanh (Nó sẽ tự hạ LR nếu cần)
    scheduler.step(val_loss)

    # 2. Logic Early Stopping (so sánh val_loss với kỷ lục best_loss)
    if val_loss < best_loss: 
        best_loss = val_loss
        best_acc = test_acc # Cập nhật điểm Acc để in ra lúc cuối
        torch.save(model.state_dict(), 'hanzi_best_weight.pth')
        epochs_no_improve = 0 # Đã phá kỷ lục -> Reset bộ đếm vi phạm về 0
        saved_msg = "⭐ Đã lưu Model (Kỷ lục Loss mới)!"
    else:
        epochs_no_improve += 1 # Không phá kỷ lục -> Ghi nhận 1 lần vi phạm
        saved_msg = f"❌ Loss không giảm ({epochs_no_improve}/{PATIENCE})"

    # =========================
    # GIAI ĐOẠN 4: IN KẾT QUẢ VÒNG LẶP
    # =========================
    epoch_time = time.time() - start_time
    current_lr = optimizer.param_groups[0]['lr'] # Lấy tốc độ học hiện tại để in ra
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] | Thời gian: {epoch_time:.0f}s | Tốc độ học (LR): {current_lr}")
    print(f"   📉 Train Loss: {train_loss:.4f} | 🎯 Train Acc: {train_acc:.2f}%")
    print(f"   📊 Val Loss:   {val_loss:.4f} | 🏆 Val Acc:   {test_acc:.2f}% -> {saved_msg}")
    print("-" * 50)

    # 3. Kích hoạt Early Stopping
    if epochs_no_improve >= PATIENCE:
        print(f"🛑 KÍCH HOẠT EARLY STOPPING! Mô hình không thể giảm Loss thêm sau {PATIENCE} epochs.")
        print(f"Tự động ngắt ở Epoch {epoch+1} để khóa chặt trạng thái tốt nhất.")
        break

print(f"🎉 ĐÃ HUẤN LUYỆN XONG! Điểm Test cao nhất (tương ứng với Loss thấp nhất): {best_acc:.2f}%")
print("Trọng số tối ưu nhất đã được lưu vào file 'hanzi_best_weight.pth'")

BẮT ĐẦU HUẤN LUYỆN (Tối đa 50 Epochs - Tích hợp ReduceLROnPlateau & Early Stopping)...
Epoch [1/50] | Thời gian: 2129s | Tốc độ học (LR): 0.001
   📉 Train Loss: 2.9064 | 🎯 Train Acc: 46.74%
   📊 Val Loss:   0.9209 | 🏆 Val Acc:   76.53% -> ⭐ Đã lưu Model (Kỷ lục Loss mới)!
--------------------------------------------------
Epoch [2/50] | Thời gian: 2682s | Tốc độ học (LR): 0.001
   📉 Train Loss: 0.8663 | 🎯 Train Acc: 77.79%
   📊 Val Loss:   0.6235 | 🏆 Val Acc:   83.65% -> ⭐ Đã lưu Model (Kỷ lục Loss mới)!
--------------------------------------------------
Epoch [3/50] | Thời gian: 2795s | Tốc độ học (LR): 0.001
   📉 Train Loss: 0.6728 | 🎯 Train Acc: 82.46%
   📊 Val Loss:   0.4534 | 🏆 Val Acc:   88.21% -> ⭐ Đã lưu Model (Kỷ lục Loss mới)!
--------------------------------------------------
Epoch [4/50] | Thời gian: 3038s | Tốc độ học (LR): 0.001
   📉 Train Loss: 0.5852 | 🎯 Train Acc: 84.65%
   📊 Val Loss:   0.4051 | 🏆 Val Acc:   89.49% -> ⭐ Đã lưu Model (Kỷ lục Loss mới)!
----------------